# Colab 02 (Full): Step LoRA Training (Inline)
- `unified_trainer.py` JSSP step 학습 워크플로우를 노트북형으로 전체화
- HF/local dataset source 지원
- resume_from_checkpoint / shuffle_data / output_dir 자동생성 / 업로드 포함


In [ ]:
!pip -q install -U unsloth
!pip -q install -U "transformers==4.57.6" huggingface_hub datasets trl peft accelerate bitsandbytes wandb


## 1) 설정


In [ ]:
import os
import csv
import random
from functools import partial
from pathlib import Path
import math
import torch
import wandb
from datasets import load_dataset
from huggingface_hub import login, hf_hub_download, HfApi, upload_folder
from unsloth import FastLanguageModel

######################
# HUGGING FACE TOKEN #
######################
HF_TOKEN = ''
######################

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = ''
if not HF_TOKEN:
    print('HF_TOKEN is empty. Fill the block above or set a Colab secret/env var before Hugging Face operations.')
else:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF login ready')

CFG = {
    # model
    'max_seq_length': 40000,
    # 'model_type': 'llama8b',
    'model_type': 'llama8b',
    # 'model_type': 'qwen25_7b',
    'dtype': 'bfloat16',
    'load_in_4bit': True,

    # LoRA
    'lora_r': 64,
    'lora_alpha': 128,
    'lora_dropout': 0.0,
    'bias': 'none',
    'use_gradient_checkpointing': 'unsloth',
    'random_state': 42,
    'use_rslora': True,
    'loftq_config': None,

    # train hparams
    'per_device_train_batch_size': 48,
    'gradient_accumulation_steps': 1,
    'per_device_eval_batch_size': 8,
    'num_train_epochs': 2,
    'learning_rate': 2e-4,
    'save_steps': 1000,
    'save_strategy': 'steps',
    'save_total_limit': 20,
    'logging_steps': 10,
    'eval_steps': 1000,
    'warmup_steps': 1000,
    'optim': 'adamw_8bit',
    'weight_decay': 0.01,
    'lr_scheduler_type': 'linear',
    'seed': 42,
    'group_by_length': True,
    'action_loss_weight': 4.0,
    'dataset_num_proc': 16,

    # speed controls
    'enable_eval': False,
    # 'max_train_samples': None,
    'max_train_samples': 1000000,
    'max_eval_samples': 512,
    'eval_subset_seed': 42,

    # task flags (unified_trainer.py 호환)
    'train_jssp': True,
    'train_fssp': False,
    'train_vrp_tsp': False,
    'train_knapsack': False,
    'train_binpack': False,
    'train_lm_head': False,
    'train_embed_tokens': False,

    # environment mode
    'env_mode': 'serial',  # serial | dispatch

    # adapter role
    'adapter_role': 'policy',  # policy | reason | mixed
    'action_code_width': 4,
    'action_code_cap': 9999,
    'step_supervision_mode': 'action_only',  # resolved automatically from adapter_role

    # unified_trainer.py 옵션 동등화
    'step_dataset_path': None,
    'fp16': None,
    'bf16': None,
    'dataloader_num_workers': 0,
    'evaluation_strategy': 'steps',
    'load_best_model_at_end': False,
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,
    'remove_unused_columns': False,
    'report_to': 'wandb',
    'run_name': None,

    # dataset source
    'dataset_source': 'hf',
    'step_dataset_hf_repo': 'HYUNJINI/jssp_policy_step_train_all_v1',
    'step_dataset_hf_file': 'train_data/jssp_step_train_policy.jsonl',
    'step_dataset_local_path': '/content/jssp_step_train_policy.jsonl',
    'policy_step_dataset_hf_repo': 'HYUNJINI/jssp_policy_step_train_all_v1',
    'policy_step_dataset_hf_file': 'train_data/jssp_step_train_policy.jsonl',
    'policy_step_dataset_local_path': '/content/jssp_step_train_policy.jsonl',
    'reason_step_dataset_hf_repo': 'HYUNJINI/jssp_reason_step_train_all_v1',
    'reason_step_dataset_hf_file': 'train_data/jssp_step_train_reason.jsonl',
    'reason_step_dataset_local_path': '/content/jssp_step_train_reason.jsonl',
    'mixed_step_dataset_hf_repo': 'HYUNJINI/jssp_mixed_step_train_all_v1',
    'mixed_step_dataset_hf_file': 'train_data/jssp_step_train_with_reason.jsonl',
    'mixed_step_dataset_local_path': '/content/jssp_step_train_with_reason.jsonl',
    'policy_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_policy_step_train_dispatch_v1',
    'policy_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_policy_dispatch.jsonl',
    'policy_step_dataset_local_path_dispatch': '/content/jssp_step_train_policy_dispatch.jsonl',
    'reason_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_reason_step_train_dispatch_v1',
    'reason_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_reason_dispatch.jsonl',
    'reason_step_dataset_local_path_dispatch': '/content/jssp_step_train_reason_dispatch.jsonl',
    'mixed_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_mixed_step_train_dispatch_v1',
    'mixed_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_with_reason_dispatch.jsonl',
    'mixed_step_dataset_local_path_dispatch': '/content/jssp_step_train_with_reason_dispatch.jsonl',

    # data options
    # 'shuffle_data': False,
    'shuffle_data': True,
    'shuffle_seed': 42,

    # output
    'output_accord': False,
    'output_list_of_lists': False,
    'output_dir': None,

    # resume
    'resume_from_checkpoint': None,

    # wandb
    'enable_wandb': False,
    'wandb_project': None,

    # optional upload
    'upload_after_train': False,
    'upload_on_save': True,
    'hf_model_repo_out': 'HYUNJINI/jssp_policy_llama8b_step_r64_ep2',
    'policy_hf_model_repo_out': 'HYUNJINI/jssp_policy_llama8b_step_r64_ep2',
    'reason_hf_model_repo_out': 'HYUNJINI/jssp_reason_llama8b_step_r64_ep2',
    'upload_source': 'latest_checkpoint',  # final | latest_checkpoint | checkpoint_tag | output_dir
    'checkpoint_tag': 'checkpoint-200',
}

MODEL_MAP = {
    'llama8b': 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    'llama1b': 'unsloth/Llama-3.1-8B-Instruct',
    'qwen25_7b': 'unsloth/Qwen3.5-9B-Base',
    'qwen25_7b_math': 'unsloth/Qwen2.5-Math-7B-Instruct-bnb-4bit',
    'qwen25_14b': 'unsloth/Qwen2.5-14B-Instruct-unsloth-bnb-4bit',
    'deepseek_8b': 'unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit',
}

print(CFG)


from transformers import TrainerCallback


def resolve_task_type(cfg):
    if cfg.get('train_jssp', False):
        return 'jssp'
    if cfg.get('train_fssp', False):
        return 'fssp'
    if cfg.get('train_vrp_tsp', False):
        return 'vrp_tsp'
    if cfg.get('train_knapsack', False):
        return 'knapsack'
    if cfg.get('train_binpack', False):
        return 'binpack'
    return 'generic'


def resolve_output_dir(cfg, task_type):
    explicit = cfg.get('output_dir')
    if explicit:
        return os.path.expanduser(str(explicit))
    env_mode = str(cfg.get('env_mode', 'serial')).lower()
    adapter_role = str(cfg.get('adapter_role', 'policy')).lower()
    model_type = str(cfg.get('model_type', 'model')).lower()
    lora_r = int(cfg.get('lora_r', 0))
    run_name = cfg.get('run_name')
    if run_name:
        folder = str(run_name)
    else:
        folder = f"{task_type}_{adapter_role}_{env_mode}_{model_type}_r{lora_r}"
    return str(Path('outputs') / folder)


def resolve_upload_dir(output_dir, upload_source='latest_checkpoint', checkpoint_tag=None):
    output_dir = Path(output_dir)
    if upload_source == 'output_dir':
        return output_dir
    if upload_source == 'final':
        final_dir = output_dir / 'final'
        return final_dir if final_dir.exists() else output_dir
    checkpoints = sorted(
        [p for p in output_dir.glob('checkpoint-*') if p.is_dir()],
        key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1,
    )
    if upload_source == 'checkpoint_tag':
        if checkpoint_tag:
            tagged = output_dir / str(checkpoint_tag)
            if tagged.exists():
                return tagged
            raise FileNotFoundError(f"checkpoint_tag not found: {tagged}")
        raise ValueError('checkpoint_tag upload_source requires CFG[\'checkpoint_tag\']')
    if checkpoints:
        return checkpoints[-1]
    final_dir = output_dir / 'final'
    if final_dir.exists():
        return final_dir
    return output_dir


def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    trainable_percent = (trainable_model_params / all_model_params * 100) if all_model_params > 0 else 0.0
    print(f"총 학습가능 매개변수: {trainable_model_params:,}개")
    print(f"총 매개변수: {all_model_params:,}개")
    print(f"학습가능 매개변수 비율: {trainable_percent:.2f}%")
    return {
        'trainable_model_params': int(trainable_model_params),
        'all_model_params': int(all_model_params),
        'trainable_percent': float(trainable_percent),
    }


class UploadCheckpointToHFCallback(TrainerCallback):
    def __init__(self, repo_id, token, output_dir, upload_source='latest_checkpoint', checkpoint_tag=None):
        self.repo_id = repo_id
        self.token = token
        self.output_dir = Path(output_dir)
        self.upload_source = upload_source
        self.checkpoint_tag = checkpoint_tag
        self.api = HfApi(token=token) if token else None

    def on_save(self, args, state, control, **kwargs):
        if not self.token:
            print('UploadCheckpointToHFCallback: HF_TOKEN empty, skip upload')
            return control
        try:
            self.api.create_repo(repo_id=self.repo_id, repo_type='model', private=False, exist_ok=True)
            folder = resolve_upload_dir(
                self.output_dir,
                upload_source=self.upload_source,
                checkpoint_tag=self.checkpoint_tag or getattr(args, 'checkpoint_tag', None),
            )
            if not folder.exists():
                print(f'UploadCheckpointToHFCallback: upload folder missing -> {folder}')
                return control
            upload_folder(
                repo_id=self.repo_id,
                repo_type='model',
                folder_path=str(folder),
                token=self.token,
                commit_message=f"Auto-upload checkpoint at step {state.global_step}",
            )
            print(f'UploadCheckpointToHFCallback: uploaded {folder} -> {self.repo_id}')
        except Exception as exc:
            print(f'UploadCheckpointToHFCallback warning: {exc}')
        return control


## 2) 유틸 함수 (inline)


In [ ]:
# --- Inline helpers aligned with source action-centric step supervision ---
import re
from typing import Any, Dict, List, Sequence

ACTION_CODE_EXTRACT_RE = re.compile(r"<\s*A\s*(\d+)\s*>", re.IGNORECASE)


def _normalize_action_code(text: str, code_width: int = 4):
    match = ACTION_CODE_EXTRACT_RE.search(str(text))
    if not match:
        return None
    return f"<A{int(match.group(1)):0{int(code_width)}d}>"


def build_step_messages(example, step_supervision_mode="action_only"):
    state_text = str(example.get("state_text", ""))
    reason_input_text = str(example.get("reason_input_text", ""))
    target_text = str(example.get("target_text", ""))
    reason_target_text = str(example.get("reason_target_text", "")).strip()

    if step_supervision_mode not in {"action_only", "action_reason", "reason_only"}:
        raise ValueError(
            f"Unsupported step_supervision_mode={step_supervision_mode}. "
            "Use one of: action_only, action_reason, reason_only."
        )

    if step_supervision_mode == "reason_only":
        user_content = reason_input_text or state_text
        if not user_content:
            raise ValueError("Missing 'reason_input_text'/'state_text' for reason dataset sample.")
        if not reason_target_text:
            legacy_reason_text = str(example.get("target_reason_text", "")).strip()
            if legacy_reason_text:
                reason_target_text = legacy_reason_text
        if not reason_target_text:
            raise ValueError("Missing 'reason_target_text' for reason dataset sample.")
        assistant_content = reason_target_text
        system_content = (
            "You are an expert JSSP scheduling analyst. "
            "Explain a fixed already-selected action. "
            "Primary objective context is final makespan (Cmax). "
            "Do not output a new action. "
            "Output format:\n"
            "Reason: ...\n"
            "Not chosen:\n"
            "- <Axxxx>: ..."
        )
    elif step_supervision_mode == "action_reason":
        if not state_text:
            raise ValueError("Missing 'state_text' for step dataset sample.")
        if not target_text:
            raise ValueError("Missing 'target_text' for step dataset sample.")
        target_action_reason_text = str(example.get("target_action_reason_text", "")).strip()
        if target_action_reason_text:
            assistant_content = target_action_reason_text
        else:
            assistant_content = f"{target_text}\n{reason_target_text}" if reason_target_text else target_text
        system_content = (
            "You are an expert JSSP scheduler. "
            "Primary objective: minimize final makespan (Cmax). "
            "At each step, choose exactly one feasible action code and explain briefly. "
            "Output format:\n"
            "<Axxxx>\n"
            "Reason: ...\n"
            "Not chosen:\n"
            "- <Axxxx>: ..."
        )
        user_content = state_text
    else:
        if not state_text:
            raise ValueError("Missing 'state_text' for step dataset sample.")
        if not target_text:
            raise ValueError("Missing 'target_text' for step dataset sample.")
        assistant_content = target_text
        system_content = (
            "You are an expert JSSP scheduler. "
            "Primary objective: minimize final makespan (Cmax). "
            "At each step, choose exactly one feasible action code. "
            "Always output exactly one code in this format: <Axxxx>."
        )
        user_content = state_text

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]


def _normalize_token_ids(tokenized_output: Any) -> List[int]:
    if tokenized_output is None:
        return []
    if hasattr(tokenized_output, "tolist"):
        tokenized_output = tokenized_output.tolist()
    if isinstance(tokenized_output, tuple):
        tokenized_output = list(tokenized_output)
    if isinstance(tokenized_output, list):
        if tokenized_output and isinstance(tokenized_output[0], list):
            if len(tokenized_output) != 1:
                raise ValueError("Expected a single tokenized sequence, got a batch.")
            tokenized_output = tokenized_output[0]
        return [int(x) for x in tokenized_output]
    raise TypeError(f"Unsupported tokenized output type: {type(tokenized_output)!r}")


def _collect_action_token_ids(tokenizer) -> List[int]:
    token_ids = []
    seen = set()
    for token in getattr(tokenizer, "additional_special_tokens", []) or []:
        parsed = _normalize_action_code(str(token))
        if parsed is None:
            continue
        token_id = tokenizer.convert_tokens_to_ids(str(token))
        if token_id is None:
            continue
        token_id = int(token_id)
        if token_id in seen:
            continue
        seen.add(token_id)
        token_ids.append(token_id)
    return token_ids


def _find_prompt_token_count(tokenizer, prompt_messages, full_ids: Sequence[int]) -> int:
    prompt_ids = _normalize_token_ids(
        tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
        )
    )
    if list(full_ids[: len(prompt_ids)]) == prompt_ids:
        return len(prompt_ids)

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    if callable(getattr(tokenizer, "__call__", None)):
        prompt_text_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        prompt_text_ids = [int(x) for x in prompt_text_ids]
        if list(full_ids[: len(prompt_text_ids)]) == prompt_text_ids:
            return len(prompt_text_ids)

    raise ValueError("Could not align prompt and assistant token boundary for step supervision.")


def _extract_action_codes(example: Dict[str, Any]) -> List[str]:
    codes: List[str] = []
    raw_codes = example.get("action_codes")
    if isinstance(raw_codes, list):
        codes.extend(str(code) for code in raw_codes if str(code).strip())
    action_code_to_job = example.get("action_code_to_job")
    if isinstance(action_code_to_job, dict):
        codes.extend(str(code) for code in action_code_to_job.keys() if str(code).strip())
    if not codes:
        state_text = str(example.get("state_text", ""))
        codes.extend(ACTION_CODE_EXTRACT_RE.findall(state_text))
        codes = [f"<A{int(code):04d}>" for code in codes]
    deduped = []
    seen = set()
    for code in codes:
        parsed = _normalize_action_code(str(code))
        if parsed is None or parsed in seen:
            continue
        seen.add(parsed)
        deduped.append(parsed)
    return deduped


def build_step_supervision_example(
    example: Dict[str, Any],
    tokenizer,
    step_supervision_mode: str = "action_only",
    max_length: int | None = None,
    action_loss_weight: float = 1.0,
):
    messages = build_step_messages(
        example=example,
        step_supervision_mode=step_supervision_mode,
    )
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    full_ids = _normalize_token_ids(
        tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
        )
    )
    prompt_len = _find_prompt_token_count(tokenizer, messages[:-1], full_ids)
    if prompt_len >= len(full_ids):
        raise ValueError("Assistant span is empty after chat template tokenization.")

    labels = [-100] * len(full_ids)
    loss_weights = [0.0] * len(full_ids)
    attention_mask = [1] * len(full_ids)
    action_target_mask = [0] * len(full_ids)
    assistant_positions = list(range(prompt_len, len(full_ids)))

    action_token_ids = set(_collect_action_token_ids(tokenizer))
    feasible_action_ids = []
    for code in _extract_action_codes(example):
        token_id = tokenizer.convert_tokens_to_ids(str(code))
        if token_id is None:
            continue
        feasible_action_ids.append(int(token_id))
    feasible_action_ids = sorted(set(feasible_action_ids))
    action_position = None
    if step_supervision_mode in {"action_only", "action_reason"}:
        for pos in assistant_positions:
            if int(full_ids[pos]) in action_token_ids:
                action_position = pos
                break
        if action_position is None:
            raise ValueError(
                "Could not find action token inside assistant span. "
                "Check action special-token installation and step targets."
            )

    if step_supervision_mode == "action_only":
        labels[action_position] = int(full_ids[action_position])
        loss_weights[action_position] = float(max(1.0, action_loss_weight))
        action_target_mask[action_position] = 1
    elif step_supervision_mode == "action_reason":
        for pos in assistant_positions:
            labels[pos] = int(full_ids[pos])
            loss_weights[pos] = 1.0
        loss_weights[action_position] = float(max(1.0, action_loss_weight))
        action_target_mask[action_position] = 1
    else:
        for pos in assistant_positions:
            labels[pos] = int(full_ids[pos])
            loss_weights[pos] = 1.0

    if max_length is not None and int(max_length) > 0 and len(full_ids) > int(max_length):
        full_ids = full_ids[: int(max_length)]
        attention_mask = attention_mask[: int(max_length)]
        labels = labels[: int(max_length)]
        loss_weights = loss_weights[: int(max_length)]
        action_target_mask = action_target_mask[: int(max_length)]

    supervised_token_count = sum(1 for label in labels if int(label) != -100)
    if supervised_token_count <= 0:
        raise ValueError(
            "No supervised assistant tokens remain after truncation. "
            "Increase max_length or shorten prompts."
        )

    out = dict(example)
    out["text"] = full_text
    out["input_ids"] = [int(x) for x in full_ids]
    out["attention_mask"] = [int(x) for x in attention_mask]
    out["labels"] = [int(x) for x in labels]
    out["loss_weights"] = [float(x) for x in loss_weights]
    out["action_target_mask"] = [int(x) for x in action_target_mask]
    out["feasible_action_ids"] = [int(x) for x in feasible_action_ids]
    out["prompt_token_count"] = int(min(prompt_len, len(full_ids)))
    out["assistant_token_count"] = int(max(0, len(full_ids) - min(prompt_len, len(full_ids))))
    out["supervised_token_count"] = int(supervised_token_count)
    return out


## 3) 학습 실행 (full)


In [ ]:
base_model_name = MODEL_MAP[CFG['model_type']]
print('base model:', base_model_name)

adapter_role = str(CFG.get('adapter_role', 'policy')).lower()
resolved_step_supervision_mode = {'policy': 'action_only', 'reason': 'reason_only', 'mixed': 'action_reason'}.get(adapter_role)
if resolved_step_supervision_mode is None:
    raise ValueError(f"Unsupported CFG['adapter_role']={adapter_role}")
CFG['step_supervision_mode'] = resolved_step_supervision_mode
if adapter_role == 'policy':
    CFG['hf_model_repo_out'] = CFG.get('policy_hf_model_repo_out', CFG['hf_model_repo_out'])
elif adapter_role == 'reason':
    CFG['hf_model_repo_out'] = CFG.get('reason_hf_model_repo_out', CFG['hf_model_repo_out'])
print('adapter_role:', adapter_role)
print('resolved step supervision mode:', resolved_step_supervision_mode)

task_type = resolve_task_type(CFG)
output_dir = Path(resolve_output_dir(CFG, task_type))
output_dir.mkdir(parents=True, exist_ok=True)

if CFG.get('step_dataset_path'):
    step_dataset_path = os.path.expanduser(CFG['step_dataset_path'])
else:
    env_mode = str(CFG.get('env_mode', 'serial')).lower()
    env_suffix = '_dispatch' if env_mode == 'dispatch' else ''
    if adapter_role == 'reason':
        dataset_repo = CFG.get(f'reason_step_dataset_hf_repo{env_suffix}', CFG.get('reason_step_dataset_hf_repo', CFG['step_dataset_hf_repo']))
        dataset_file = CFG.get(f'reason_step_dataset_hf_file{env_suffix}', CFG.get('reason_step_dataset_hf_file', CFG['step_dataset_hf_file']))
        dataset_local_path = CFG.get(f'reason_step_dataset_local_path{env_suffix}', CFG.get('reason_step_dataset_local_path', CFG['step_dataset_local_path']))
    elif adapter_role == 'mixed':
        dataset_repo = CFG.get(f'mixed_step_dataset_hf_repo{env_suffix}', CFG.get('mixed_step_dataset_hf_repo', CFG['step_dataset_hf_repo']))
        dataset_file = CFG.get(f'mixed_step_dataset_hf_file{env_suffix}', CFG.get('mixed_step_dataset_hf_file', CFG['step_dataset_hf_file']))
        dataset_local_path = CFG.get(f'mixed_step_dataset_local_path{env_suffix}', CFG.get('mixed_step_dataset_local_path', CFG['step_dataset_local_path']))
    else:
        dataset_repo = CFG.get(f'policy_step_dataset_hf_repo{env_suffix}', CFG.get('policy_step_dataset_hf_repo', CFG['step_dataset_hf_repo']))
        dataset_file = CFG.get(f'policy_step_dataset_hf_file{env_suffix}', CFG.get('policy_step_dataset_hf_file', CFG['step_dataset_hf_file']))
        dataset_local_path = CFG.get(f'policy_step_dataset_local_path{env_suffix}', CFG.get('policy_step_dataset_local_path', CFG['step_dataset_local_path']))

    if CFG['dataset_source'] == 'hf':
        step_dataset_path = hf_hub_download(
            repo_id=dataset_repo,
            repo_type='dataset',
            filename=dataset_file,
            token=HF_TOKEN,
        )
    elif CFG['dataset_source'] == 'local':
        step_dataset_path = dataset_local_path
    else:
        raise ValueError("CFG['dataset_source'] must be 'hf' or 'local'")

print('step dataset:', step_dataset_path)

dtype = torch.bfloat16 if CFG['dtype'] == 'bfloat16' else torch.float16
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=CFG['max_seq_length'],
    load_in_4bit=CFG['load_in_4bit'],
    dtype=dtype,
    local_files_only=False,
)

modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
if CFG['train_lm_head']:
    modules.append('lm_head')
if CFG['train_embed_tokens']:
    modules.append('embed_tokens')
model = FastLanguageModel.get_peft_model(
    model,
    r=CFG['lora_r'],
    target_modules=modules,
    lora_alpha=CFG['lora_alpha'],
    lora_dropout=CFG['lora_dropout'],
    bias=CFG['bias'],
    use_rslora=CFG['use_rslora'],
    use_gradient_checkpointing=CFG['use_gradient_checkpointing'],
    random_state=CFG['random_state'],
    loftq_config=CFG['loftq_config'],
)
print_number_of_trainable_model_parameters(model)

from datasets import Dataset
import json

REQUIRED_STEP_KEYS = [
    'instance_id',
    'source_index',
    'state_text',
    'reason_input_text',
    'target_text',
    'target_action_reason_text',
    'target_reason_text',
    'reason_target_text',
    'feature_schema_version',
    'num_jobs',
    'num_machines',
    'total_steps',
    'step_idx',
]


def _normalize_step_row(row):
    out = {}
    out['instance_id'] = str(row.get('instance_id', '') or '')
    source_index = row.get('source_index', -1)
    out['source_index'] = int(source_index) if source_index is not None else -1
    out['state_text'] = str(row.get('state_text', ''))
    out['reason_input_text'] = str(row.get('reason_input_text', ''))
    out['target_text'] = str(row.get('target_text', ''))
    out['target_action_reason_text'] = str(row.get('target_action_reason_text', ''))
    out['target_reason_text'] = str(row.get('target_reason_text', ''))
    out['reason_target_text'] = str(row.get('reason_target_text', ''))
    out['feature_schema_version'] = str(row.get('feature_schema_version', 'unknown'))
    out['num_jobs'] = int(row.get('num_jobs', 0))
    out['num_machines'] = int(row.get('num_machines', 0))
    out['total_steps'] = int(row.get('total_steps', 0))
    out['step_idx'] = int(row.get('step_idx', 0))
    return out


def _iter_step_rows(path):
    path = os.path.expanduser(str(path))
    with open(path, 'r', encoding='utf-8') as f:
        first_non_ws = None
        while True:
            ch = f.read(1)
            if ch == '':
                break
            if not ch.isspace():
                first_non_ws = ch
                break
        f.seek(0)

        if first_non_ws == '[':
            data = json.load(f)
            if not isinstance(data, list):
                raise ValueError('JSON input must be a list when using array format.')
            for row in data:
                yield _normalize_step_row(row)
        else:
            for line_idx, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                except Exception as e:
                    raise ValueError(f'Invalid JSONL at line {line_idx}: {e}')
                yield _normalize_step_row(row)


def _resolve_instance_keys(ds):
    columns = set(ds.column_names)
    instance_values = ds['instance_id'] if 'instance_id' in columns else None
    source_values = ds['source_index'] if 'source_index' in columns else None
    keys = []
    for row_idx in range(len(ds)):
        instance_id = ''
        if instance_values is not None:
            raw_instance_id = instance_values[row_idx]
            if raw_instance_id is not None:
                instance_id = str(raw_instance_id).strip()
        if instance_id:
            keys.append(instance_id)
            continue
        if source_values is not None:
            keys.append(f"source_{int(source_values[row_idx])}")
            continue
        raise ValueError("Step dataset must contain either 'instance_id' or 'source_index' for instance-level split.")
    return keys


def _ordered_unique(keys):
    seen = set()
    ordered = []
    for key in keys:
        if key in seen:
            continue
        seen.add(key)
        ordered.append(key)
    return ordered


def _split_indices_by_instance(ds, test_ratio, split_seed, enable_eval_split):
    instance_keys = _resolve_instance_keys(ds)
    ordered_instances = _ordered_unique(instance_keys)
    if not ordered_instances:
        raise ValueError('No instances found for instance-level split.')

    if not enable_eval_split:
        return list(range(len(ds))), [], ordered_instances, []

    shuffled_instances = ordered_instances[:]
    rng = random.Random(int(split_seed))
    rng.shuffle(shuffled_instances)
    eval_instance_count = max(1, int(round(len(shuffled_instances) * float(test_ratio))))
    eval_instance_count = min(eval_instance_count, max(1, len(shuffled_instances) - 1))
    eval_instance_set = set(shuffled_instances[:eval_instance_count])

    train_indices = []
    eval_indices = []
    for row_idx, instance_key in enumerate(instance_keys):
        if instance_key in eval_instance_set:
            eval_indices.append(row_idx)
        else:
            train_indices.append(row_idx)

    train_instance_ids = [iid for iid in ordered_instances if iid not in eval_instance_set]
    eval_instance_ids = [iid for iid in ordered_instances if iid in eval_instance_set]
    return train_indices, eval_indices, train_instance_ids, eval_instance_ids


dataset = Dataset.from_generator(_iter_step_rows, gen_kwargs={'path': step_dataset_path})
print(f'raw dataset rows: {len(dataset):,}')

enable_eval = bool(CFG.get('enable_eval', True))
test_size = 0.05
train_indices, eval_indices, train_instance_ids, eval_instance_ids = _split_indices_by_instance(
    dataset,
    test_ratio=test_size,
    split_seed=CFG['seed'],
    enable_eval_split=enable_eval,
)

train_dataset = dataset.select(train_indices)
eval_dataset = dataset.select(eval_indices) if enable_eval else None

overlap_count = len(set(train_instance_ids) & set(eval_instance_ids))
print(
    'instance split:',
    f"train_instances={len(train_instance_ids):,}",
    f"eval_instances={len(eval_instance_ids):,}",
    f"overlap={overlap_count}",
)
print(
    'row counts before map:',
    f"train_rows={len(train_dataset):,}",
    f"eval_rows={(len(eval_dataset) if eval_dataset is not None else 0):,}",
)

if CFG['shuffle_data']:
    print(f"shuffle train split enabled (seed={CFG['shuffle_seed']})")
    train_dataset = train_dataset.shuffle(seed=CFG['shuffle_seed'])
else:
    print('shuffle train split disabled')

if CFG.get('max_train_samples') is not None:
    train_cap = min(int(CFG['max_train_samples']), len(train_dataset))
    print('train sample cap:', train_cap)
    train_dataset = train_dataset.select(range(train_cap))

if eval_dataset is not None and CFG.get('max_eval_samples') is not None:
    eval_cap = min(int(CFG['max_eval_samples']), len(eval_dataset))
    print('eval sample cap:', eval_cap)
    eval_dataset = eval_dataset.shuffle(seed=int(CFG.get('eval_subset_seed', 42))).select(range(eval_cap))

fmt = partial(
    build_step_supervision_example,
    tokenizer=tokenizer,
    step_supervision_mode=resolved_step_supervision_mode,
    max_length=CFG['max_seq_length'],
    action_loss_weight=float(CFG.get('action_loss_weight', 4.0)),
)
step_map_num_proc = max(1, int(CFG.get('dataset_num_proc', 1)))
print('step supervision map num_proc:', step_map_num_proc)
train_dataset = train_dataset.map(fmt, num_proc=step_map_num_proc)
if eval_dataset is not None:
    eval_dataset = eval_dataset.map(fmt, num_proc=step_map_num_proc)

enable_eval = bool(CFG.get('enable_eval', True))
eval_strategy_value = CFG['evaluation_strategy'] if enable_eval else 'no'
eval_steps_value = CFG['eval_steps'] if enable_eval else None
eval_dataset_for_trainer = eval_dataset if enable_eval else None

print('train rows:', len(train_dataset))
print('eval rows:', len(eval_dataset) if eval_dataset is not None else 0)
if len(train_dataset):
    sample_feature_schema = train_dataset[0].get('feature_schema_version', 'unknown') if hasattr(train_dataset[0], 'get') else 'unknown'
    print('feature schema:', sample_feature_schema)
    print('sample prompt\n', train_dataset[0]['text'][:500])
    print('sample supervision stats', {
        'prompt_tokens': train_dataset[0].get('prompt_token_count'),
        'assistant_tokens': train_dataset[0].get('assistant_token_count'),
        'supervised_tokens': train_dataset[0].get('supervised_token_count'),
    })

print('dataset preprocessing ready; run the next cell to start training')





In [ ]:
print('starting training cell; re-run previous cell if you changed dataset/preprocessing settings or want a fresh model state')
import inspect
import torch.nn as nn

class StepSupervisionCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    def __call__(self, features):
        max_len = max(len(feature['input_ids']) for feature in features)
        max_action_count = max(len(feature.get('feasible_action_ids', [])) for feature in features)
        batch = {
            'input_ids': [],
            'attention_mask': [],
            'labels': [],
            'loss_weights': [],
            'action_target_mask': [],
            'feasible_action_ids': [],
        }
        for feature in features:
            seq_len = len(feature['input_ids'])
            pad_len = max_len - seq_len
            batch['input_ids'].append(list(feature['input_ids']) + [int(self.pad_token_id)] * pad_len)
            batch['attention_mask'].append(list(feature['attention_mask']) + [0] * pad_len)
            batch['labels'].append(list(feature['labels']) + [-100] * pad_len)
            batch['loss_weights'].append(list(feature['loss_weights']) + [0.0] * pad_len)
            batch['action_target_mask'].append(list(feature.get('action_target_mask', [])) + [0] * pad_len)
            feasible_action_ids = list(feature.get('feasible_action_ids', []))
            batch['feasible_action_ids'].append(feasible_action_ids + [-1] * (max_action_count - len(feasible_action_ids)))
        return {
            'input_ids': torch.tensor(batch['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(batch['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(batch['labels'], dtype=torch.long),
            'loss_weights': torch.tensor(batch['loss_weights'], dtype=torch.float32),
            'action_target_mask': torch.tensor(batch['action_target_mask'], dtype=torch.long),
            'feasible_action_ids': torch.tensor(batch['feasible_action_ids'], dtype=torch.long),
        }


def _build_step_supervision_trainer(base_cls):
    class StepSupervisionTrainer(base_cls):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            model_inputs = dict(inputs)
            labels = model_inputs.pop('labels')
            loss_weights = model_inputs.pop('loss_weights', None)
            action_target_mask = model_inputs.pop('action_target_mask', None)
            feasible_action_ids = model_inputs.pop('feasible_action_ids', None)
            outputs = model(**model_inputs)
            logits = outputs.get('logits') if isinstance(outputs, dict) else outputs.logits

            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            loss_fct = nn.CrossEntropyLoss(ignore_index=-100, reduction='none')
            token_loss = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            ).view_as(shift_labels)

            valid_mask = shift_labels.ne(-100)
            if loss_weights is None:
                weights = valid_mask.to(token_loss.dtype)
            else:
                shift_weights = loss_weights[..., 1:].contiguous().to(token_loss.dtype)
                weights = shift_weights * valid_mask.to(token_loss.dtype)

            if action_target_mask is not None:
                shift_action_mask = action_target_mask[..., 1:].contiguous().bool()
            else:
                shift_action_mask = torch.zeros_like(valid_mask, dtype=torch.bool)

            base_weights = weights * (~shift_action_mask).to(weights.dtype)
            loss_num = (token_loss * base_weights).sum()
            denom = base_weights.sum()

            if feasible_action_ids is not None and bool(shift_action_mask.any().item()):
                action_rows, action_cols = torch.nonzero(shift_action_mask, as_tuple=True)
                for row_idx, col_idx in zip(action_rows.tolist(), action_cols.tolist()):
                    row_feasible_ids = feasible_action_ids[row_idx]
                    row_feasible_ids = row_feasible_ids[row_feasible_ids.ge(0)]
                    if row_feasible_ids.numel() <= 0:
                        continue
                    target_id = int(shift_labels[row_idx, col_idx].item())
                    target_matches = torch.nonzero(row_feasible_ids.eq(target_id), as_tuple=False).view(-1)
                    if target_matches.numel() <= 0:
                        continue
                    candidate_logits = shift_logits[row_idx, col_idx].index_select(
                        0,
                        row_feasible_ids.to(shift_logits.device, dtype=torch.long),
                    )
                    target_index = target_matches[0].to(device=shift_logits.device, dtype=torch.long).view(1)
                    action_loss = nn.functional.cross_entropy(
                        candidate_logits.view(1, -1),
                        target_index,
                        reduction='sum',
                    )
                    action_weight = weights[row_idx, col_idx].to(action_loss.dtype)
                    loss_num = loss_num + (action_loss * action_weight)
                    denom = denom + action_weight

            denom = denom.clamp_min(1.0)
            loss = loss_num / denom
            return (loss, outputs) if return_outputs else loss

    return StepSupervisionTrainer

if CFG['enable_wandb']:
    project_name = CFG['wandb_project'] or f"{task_type}_optimization"
    wandb.init(project=project_name, name=f"{task_type}_{CFG['model_type']}_r{CFG['lora_r']}", config=CFG)
else:
    os.environ['WANDB_MODE'] = 'disabled'

with open(output_dir / 'training_hyperparams_args.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    for k, v in CFG.items():
        writer.writerow([k, v])

if CFG['train_lm_head'] and CFG['train_embed_tokens']:
    from unsloth import UnslothTrainer, UnslothTrainingArguments
    Trainer = UnslothTrainer
    TrainingArgs = UnslothTrainingArguments
    print('Training with UnslothTrainer')
else:
    from transformers import Trainer as HFTrainer, TrainingArguments
    Trainer = HFTrainer
    TrainingArgs = TrainingArguments
    print('Training with Trainer')

Trainer = _build_step_supervision_trainer(Trainer)
data_collator = StepSupervisionCollator(tokenizer=tokenizer)

effective_bf16 = CFG['bf16'] if CFG['bf16'] is not None else (True if CFG['dtype'] == 'bfloat16' else False)
effective_fp16 = CFG['fp16'] if CFG['fp16'] is not None else (True if CFG['dtype'] == 'float16' else False)

training_args_kwargs = {
    'per_device_train_batch_size': CFG['per_device_train_batch_size'],
    'gradient_accumulation_steps': CFG['gradient_accumulation_steps'],
    'warmup_steps': CFG['warmup_steps'],
    'num_train_epochs': CFG['num_train_epochs'],
    'learning_rate': CFG['learning_rate'],
    'bf16': effective_bf16,
    'fp16': effective_fp16,
    'logging_steps': CFG['logging_steps'],
    'optim': CFG['optim'],
    'weight_decay': CFG['weight_decay'],
    'lr_scheduler_type': CFG['lr_scheduler_type'],
    'seed': CFG['seed'],
    'output_dir': str(output_dir),
    'report_to': (CFG['report_to'] if CFG['enable_wandb'] else 'none'),
    'load_best_model_at_end': (CFG['load_best_model_at_end'] if enable_eval else False),
    'metric_for_best_model': CFG['metric_for_best_model'],
    'greater_is_better': CFG['greater_is_better'],
    'save_total_limit': CFG['save_total_limit'],
    'save_steps': CFG['save_steps'],
    'save_strategy': CFG['save_strategy'],
    'eval_strategy': eval_strategy_value,
    'eval_steps': eval_steps_value,
    'per_device_eval_batch_size': CFG['per_device_eval_batch_size'],
    'group_by_length': CFG['group_by_length'],
    'dataloader_num_workers': CFG['dataloader_num_workers'],
    'remove_unused_columns': CFG['remove_unused_columns'],
    'run_name': CFG['run_name'],
}

supported_training_arg_names = {
    name for name in inspect.signature(TrainingArgs.__init__).parameters.keys()
    if name != 'self'
}

if 'eval_strategy' in training_args_kwargs and 'evaluation_strategy' in supported_training_arg_names:
    training_args_kwargs['evaluation_strategy'] = training_args_kwargs.pop('eval_strategy')
elif 'evaluation_strategy' in training_args_kwargs and 'eval_strategy' in supported_training_arg_names:
    training_args_kwargs['eval_strategy'] = training_args_kwargs.pop('evaluation_strategy')

final_training_args = {
    k: v for k, v in training_args_kwargs.items()
    if v is not None and k in supported_training_arg_names
}
dropped_training_args = sorted(
    k for k, v in training_args_kwargs.items()
    if v is not None and k not in supported_training_arg_names
)
if dropped_training_args:
    print('TrainingArgs unsupported keys dropped:', dropped_training_args)
print('resolved training args subset:', {
    'per_device_train_batch_size': final_training_args.get('per_device_train_batch_size'),
    'gradient_accumulation_steps': final_training_args.get('gradient_accumulation_steps'),
    'per_device_eval_batch_size': final_training_args.get('per_device_eval_batch_size'),
    'num_train_epochs': final_training_args.get('num_train_epochs'),
    'learning_rate': final_training_args.get('learning_rate'),
    'save_steps': final_training_args.get('save_steps'),
})

callbacks = []
if CFG.get('upload_on_save', False):
    callbacks.append(
        UploadCheckpointToHFCallback(
            repo_id=CFG['hf_model_repo_out'],
            token=HF_TOKEN,
            output_dir=output_dir,
            upload_source=CFG.get('upload_source', 'latest_checkpoint'),
        )
    )
    print(f"checkpoint auto-upload enabled -> {CFG['hf_model_repo_out']}")
else:
    print('checkpoint auto-upload disabled')

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset_for_trainer,
    data_collator=data_collator,
    args=TrainingArgs(**final_training_args),
    callbacks=callbacks if callbacks else None,
)
print('🎯 action-centric supervision:', {
    'mode': resolved_step_supervision_mode,
    'action_loss_weight': float(CFG.get('action_loss_weight', 4.0)),
    'supervision': 'assistant-only/action-targeted+feasible-set-ce',
})
print('DEBUG: trainer object created')
resume_arg = None

if CFG['resume_from_checkpoint']:
    checkpoint_path = os.path.expanduser(CFG['resume_from_checkpoint'])
    if os.path.exists(checkpoint_path):
        print('resume from checkpoint:', checkpoint_path)
        resume_arg = checkpoint_path
    else:
        print('checkpoint not found, train from scratch:', checkpoint_path)
else:
    contains_checkpoints = any('checkpoint' in p.name and p.is_dir() for p in output_dir.iterdir())
    if contains_checkpoints:
        print('checkpoint detected in output_dir, continue training')
        resume_arg = True
    else:
        print('no checkpoint in output_dir, train from scratch')

print('DEBUG: building train dataloader', flush=True)
train_dataloader = trainer.get_train_dataloader()
print('DEBUG: train dataloader ready', flush=True)

print('DEBUG: fetching first batch', flush=True)
first_batch = next(iter(train_dataloader))
debug_batch_shapes = {}
for k, v in first_batch.items():
    if hasattr(v, 'shape'):
        debug_batch_shapes[k] = tuple(v.shape)
    else:
        debug_batch_shapes[k] = type(v).__name__
print('DEBUG: first batch fetched', debug_batch_shapes, flush=True)

del first_batch
del train_dataloader
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('DEBUG: entering trainer.train()', 'resume_arg=', resume_arg, flush=True)

if resume_arg is None:
    trainer.train()
else:
    trainer.train(resume_from_checkpoint=resume_arg)

final_dir = output_dir / 'final'
final_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

print('training done')
print('output_dir:', output_dir)
print('final_dir:', final_dir)
print('checkpoints:')
for p in sorted(output_dir.glob('checkpoint-*')):
    print(' -', p)



## 4) (선택) 학습 완료 모델 HF 업로드


In [ ]:
if CFG['upload_after_train']:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=CFG['hf_model_repo_out'], repo_type='model', private=False, exist_ok=True)

    output_dir = Path(resolve_output_dir(CFG, resolve_task_type(CFG)))
    upload_dir = resolve_upload_dir(output_dir, CFG['upload_source'], CFG['checkpoint_tag'])
    print('upload_dir:', upload_dir)

    upload_folder(
        repo_id=CFG['hf_model_repo_out'],
        repo_type='model',
        folder_path=str(upload_dir),
        token=HF_TOKEN,
        commit_message=f"Upload LoRA ({upload_dir.name}) from colab_02_full",
    )

    files = api.list_repo_files(repo_id=CFG['hf_model_repo_out'], repo_type='model')
    print(f"upload done: {CFG['hf_model_repo_out']} ({len(files)} files)")
    for f in files:
        print(' -', f)
else:
    print('CFG[upload_after_train]=False, skip upload')
